In [29]:
import sqlite3
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
import io
import re
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def initialize_database(db_name="big_brother.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS Seasons (
        season_id TEXT PRIMARY KEY,
        year INTEGER,
        has_blockbuster BOOLEAN
    )""")

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS Houseguests (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player_name TEXT,
        season_id TEXT,
        occupation TEXT,
        age INTEGER,
        UNIQUE(player_name, season_id),
        FOREIGN KEY (season_id) REFERENCES Seasons(season_id)
    )""")

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS Competitions_Won (
        win_id INTEGER PRIMARY KEY AUTOINCREMENT,
        comp_code TEXT UNIQUE,
        player_id INTEGER,
        season_id TEXT,
        comp_type TEXT CHECK(comp_type IN ('HOH', 'Veto', 'BlockBuster')),
        points INTEGER,
        FOREIGN KEY (player_id) REFERENCES Houseguests(player_id),
        FOREIGN KEY (season_id) REFERENCES Seasons(season_id)
    )""")

    conn.commit()
    return conn

def clean_text(text_str):
    text = str(text_str)
    text = text.replace('\n', ' ')
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)

    # Truncate everything starting from returnee markers like "Big Brother", "BB", or "&"
    text = re.split(r'(?:Big Brother|BB\s*\d+|&)', text, flags=re.IGNORECASE)[0]

    return text.strip()

def is_valid_player_name(name):
    if not name:
        return False
    if any(p in name for p in ['.', '!', '?', ';']):
        return False
    if len(name) > 30 or len(name.split()) > 4:
        return False
    if name.lower() in ['none', 'nan', '', 'evicted', 'winner', 'runner-up']:
        return False
    return True

def parse_houseguests_table(soup, season_id, cursor):
    name_mapping = {}
    tables = soup.find_all('table', {'class': 'wikitable'})
    hg_table = None

    for t in tables:
        t_text = t.get_text()
        if 'Occupation' in t_text and 'Residence' in t_text:
            hg_table = t
            break

    if not hg_table:
        return name_mapping

    try:
        df = pd.read_html(io.StringIO(str(hg_table)))[0]
        df.columns = [str(c).strip().lower() for c in df.columns]

        name_col = next((c for c in df.columns if 'name' in c), None)
        occ_col = next((c for c in df.columns if 'occupation' in c), None)
        age_col = next((c for c in df.columns if 'age' in c), None)

        if not name_col:
            return name_mapping

        for _, row in df.iterrows():
            raw_name = str(row[name_col])
            cleaned_name = clean_text(raw_name)

            if not is_valid_player_name(cleaned_name):
                continue

            occupation = clean_text(row[occ_col]) if occ_col else None
            try:
                age = int(re.sub(r'\D', '', str(row[age_col]))) if age_col else None
            except:
                age = None

            full_name = re.sub(r'\s+', ' ', cleaned_name).strip()

            cursor.execute("""
                INSERT OR IGNORE INTO Houseguests (player_name, season_id, occupation, age)
                VALUES (?, ?, ?, ?)
            """, (full_name, season_id, occupation, age))

            cursor.execute("SELECT player_id FROM Houseguests WHERE player_name = ? AND season_id = ?", (full_name, season_id))
            p_res = cursor.fetchone()
            if not p_res:
                continue
            player_id = p_res[0]

            name_mapping[full_name.lower()] = player_id
            for part in full_name.lower().split():
                name_mapping[part] = player_id

            quotes = re.findall(r'"([^"]*)"', raw_name)
            for q in quotes:
                name_mapping[q.lower()] = player_id

    except Exception as e:
        print(f"[-] Error parsing houseguests table for {season_id}: {e}")

    return name_mapping

def import_seasons_from_wikipedia(conn, start_season=2, end_season=28):
    cursor = conn.cursor()
    points_map = {'HOH': 5, 'Veto': 3, 'BlockBuster': 1}

    for season_num in range(start_season, end_season + 1):
        season_id = f"BB{season_num}"
        url = f"https://en.wikipedia.org/wiki/Big_Brother_{season_num}_(American_season)"

        print(f"Scraping {season_id}...")
        try:
            response = requests.get(url, headers=HEADERS)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            name_to_id_map = parse_houseguests_table(soup, season_id, cursor)
            conn.commit()

            tables = soup.find_all('table', {'class': 'wikitable'})
            table_elem = None
            for t in tables:
                t_text = t.get_text()
                if 'Head of' in t_text and ('Nominated' in t_text or 'Evicted' in t_text):
                    table_elem = t
                    break

            if not table_elem:
                print(f"[-] Voting history table not found for {season_id}.")
                continue

            parsed_tables = pd.read_html(io.StringIO(str(table_elem)))
            voting_table = parsed_tables[0]

        except Exception as e:
            print(f"[-] Could not fetch or parse URL for {season_id}: {e}")
            continue

        has_bb = 1 if season_num >= 26 else 0
        cursor.execute("INSERT OR IGNORE INTO Seasons (season_id, has_blockbuster) VALUES (?, ?)", (season_id, has_bb))
        conn.commit()

        if isinstance(voting_table.columns, pd.MultiIndex):
            voting_table.columns = [' '.join(str(c) for c in col).strip() for col in voting_table.columns.values]
        else:
            voting_table.columns = [str(c).strip() for c in voting_table.columns]

        for _, row in voting_table.iterrows():
            raw_label = str(row.iloc[0])
            row_label_clean = clean_text(raw_label).lower()
            row_label_no_space = row_label_clean.replace(" ", "")

            is_hoh_row = 'headofhousehold' in row_label_no_space
            is_veto_row = 'vetowinner' in row_label_no_space or 'powerofveto' in row_label_no_space

            # FIXED: Space-insensitive check catches "blockbuster", "block buster", or "ai arena"
            is_bb_row = 'blockbuster' in row_label_no_space or 'aiana' in row_label_no_space

            if not (is_hoh_row or is_veto_row or is_bb_row):
                continue

            for col_idx in range(1, len(row)):
                cell_value = clean_text(row.iloc[col_idx])
                if not is_valid_player_name(cell_value):
                    continue

                cell_key = cell_value.lower()
                player_id = name_to_id_map.get(cell_key)

                if not player_id:
                    for k, pid in name_to_id_map.items():
                        if cell_key in k or k in cell_key:
                            player_id = pid
                            break

                if not player_id:
                    continue

                if is_hoh_row and season_num >= 2:
                    comp_type = 'HOH'
                elif is_veto_row and season_num >= 3:
                    comp_type = 'Veto'
                elif is_bb_row and season_num >= 26:
                    comp_type = 'BlockBuster'
                else:
                    continue

                week_num = col_idx
                comp_code = f"{season_id}-W{week_num:02d}-{comp_type}"

                try:
                    cursor.execute("""
                        INSERT OR IGNORE INTO Competitions_Won (comp_code, player_id, season_id, comp_type, points)
                        VALUES (?, ?, ?, ?, ?)
                    """, (comp_code, player_id, season_id, comp_type, points_map[comp_type]))
                except sqlite3.IntegrityError:
                    pass

        conn.commit()
        print(f"[+] Successfully parsed and loaded {season_id}")
        time.sleep(1)

# Rebuild database up to BB28
if os.path.exists("big_brother.db"):
    os.remove("big_brother.db")
db = initialize_database()
import_seasons_from_wikipedia(db, start_season=2, end_season=28)
db.close()
print("Database build complete with BlockBuster space-insensitive matching fixed!")

Scraping BB2...
[+] Successfully parsed and loaded BB2
Scraping BB3...
[+] Successfully parsed and loaded BB3
Scraping BB4...
[+] Successfully parsed and loaded BB4
Scraping BB5...
[+] Successfully parsed and loaded BB5
Scraping BB6...
[+] Successfully parsed and loaded BB6
Scraping BB7...
[+] Successfully parsed and loaded BB7
Scraping BB8...
[+] Successfully parsed and loaded BB8
Scraping BB9...
[+] Successfully parsed and loaded BB9
Scraping BB10...
[+] Successfully parsed and loaded BB10
Scraping BB11...
[+] Successfully parsed and loaded BB11
Scraping BB12...
[+] Successfully parsed and loaded BB12
Scraping BB13...
[+] Successfully parsed and loaded BB13
Scraping BB14...
[+] Successfully parsed and loaded BB14
Scraping BB15...
[+] Successfully parsed and loaded BB15
Scraping BB16...
[+] Successfully parsed and loaded BB16
Scraping BB17...
[+] Successfully parsed and loaded BB17
Scraping BB18...
[+] Successfully parsed and loaded BB18
Scraping BB19...
[+] Successfully parsed and lo

In [33]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect("big_brother.db")

# 1. All-Time Top 10 Leaderboard (Combines stats if a player returned in multiple seasons)
all_time_query = """
    SELECT h.player_name AS "Houseguest",
           GROUP_CONCAT(DISTINCT c.season_id) AS "Seasons",
           SUM(CASE WHEN c.comp_type = 'HOH' THEN 1 ELSE 0 END) AS "HOH Wins",
           SUM(CASE WHEN c.comp_type = 'Veto' THEN 1 ELSE 0 END) AS "Veto Wins",
           SUM(CASE WHEN c.comp_type = 'BlockBuster' THEN 1 ELSE 0 END) AS "BlockBuster Wins",
           SUM(c.points) AS "Total Points"
    FROM Competitions_Won c
    JOIN Houseguests h ON c.player_id = h.player_id
    GROUP BY h.player_name
    ORDER BY "Total Points" DESC
    LIMIT 10;
"""

df_all_time = pd.read_sql(all_time_query, conn)

# 2. Current Season (BB28) Leaderboard
target_season = "BB28"
season_query = f"""
    SELECT h.player_name AS "Houseguest",
           h.occupation AS "Occupation",
           SUM(CASE WHEN c.comp_type = 'HOH' THEN 1 ELSE 0 END) AS "HOH Wins",
           SUM(CASE WHEN c.comp_type = 'Veto' THEN 1 ELSE 0 END) AS "Veto Wins",
           SUM(CASE WHEN c.comp_type = 'BlockBuster' THEN 1 ELSE 0 END) AS "BlockBuster Wins",
           SUM(c.points) AS "Total Points"
    FROM Competitions_Won c
    JOIN Houseguests h ON c.player_id = h.player_id
    WHERE c.season_id = '{target_season}'
    GROUP BY h.player_id
    ORDER BY "Total Points" DESC;
"""

df_season = pd.read_sql(season_query, conn)

# 3. All Time single season Leaderboard
# All-Time Single-Season Top 10 Leaderboard
single_season_query = """
    SELECT h.player_name AS "Houseguest",
           c.season_id AS "Season",
           h.occupation AS "Occupation",
           SUM(CASE WHEN c.comp_type = 'HOH' THEN 1 ELSE 0 END) AS "HOH Wins",
           SUM(CASE WHEN c.comp_type = 'Veto' THEN 1 ELSE 0 END) AS "Veto Wins",
           SUM(CASE WHEN c.comp_type = 'BlockBuster' THEN 1 ELSE 0 END) AS "BlockBuster Wins",
           SUM(c.points) AS "Total Points"
    FROM Competitions_Won c
    JOIN Houseguests h ON c.player_id = h.player_id
    GROUP BY h.player_id
    ORDER BY "Total Points" DESC
    LIMIT 10;
"""
df_single_season = pd.read_sql(single_season_query, conn)

conn.close()

# Display the results cleanly
print("🏆 ALL-TIME TOP 10 COMP BEASTS")
display(df_all_time)

print(f"\n🌟 {target_season} COMP BEAST LEADERBOARD")
display(df_season)

print("\n🏆 ALL-TIME TOP 10 SINGLE-SEASON PERFORMANCES")
display(df_single_season)

🏆 ALL-TIME TOP 10 COMP BEASTS


,Houseguest,Seasons,HOH Wins,Veto Wins,BlockBuster Wins,Total Points
0,Cody Calafiore,"BB16,BB22",7,7,0,56
1,Rachel Reilly,"BB12,BB13,BB27",7,2,0,41
2,Tyler Crispen,"BB20,BB22",5,5,0,40
3,Jag Bains,BB25,3,7,0,36
4,Daniele Donato,"BB8,BB13",4,5,0,35
5,Monte Taylor,BB24,5,2,0,31
6,Makensy Manbeck,BB26,3,5,0,30
7,Vanessa Rousso,BB17,4,3,0,29
8,Dan Gheesling,"BB10,BB14",4,3,0,29
9,"Robert ""Memphis"" Garrett","BB10,BB22",3,4,0,27



🌟 BB28 COMP BEAST LEADERBOARD


,Houseguest,Occupation,HOH Wins,Veto Wins,BlockBuster Wins,Total Points
0,Yash Patel,Finance analyst,2,2,1,17
1,Rick Devens,Communications director,1,2,2,13
2,Drew Campbell,Surgical dental assistant,1,1,2,10
3,Dee Valladares,Entrepreneur,2,0,0,10
4,"Kamuela ""Kamu"" Kirk",MMA fighter,1,1,0,8
5,La Trice Verrett,Boutique salesperson,1,1,0,8
6,Haley Thogmartin,Telemedicine executive,1,0,1,6
7,Barrett Pfeiffer,Jumbotron engineer,1,0,0,5
8,Mallory Aurichio,Rocket scientist,0,1,1,4
9,Lyric Medeiros,Attorney,0,1,0,3



🏆 ALL-TIME TOP 10 SINGLE-SEASON PERFORMANCES


,Houseguest,Season,Occupation,HOH Wins,Veto Wins,BlockBuster Wins,Total Points
0,Jag Bains,BB25,Truck company owner,3,7,0,36
1,Cody Calafiore,BB22,Soccer coach,4,4,0,32
2,Monte Taylor,BB24,Personal trainer,5,2,0,31
3,Makensy Manbeck,BB26,Construction project manager,3,5,0,30
4,Vanessa Rousso,BB17,Professional poker player,4,3,0,29
5,Michael Bruner,BB24,Attorney,3,4,0,27
6,Steve Moses,BB17,College student,4,2,0,26
7,Ian Terry,BB14,Engineering student,4,2,0,26
8,Rachel Reilly,BB13,Event hostess,4,2,0,26
9,Morgan Pope,BB27,Gamer,2,5,0,25
